# 02. Índice y recuperación oficial

Construir el índice elegido y recuperar evidencia para las 50 preguntas. El conjunto recuperado se guarda una vez para usarlo en la evaluación inicial y en la comparación de decoders.

## Entorno

El mismo notebook corre en local o en Colab.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import zipfile


os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

MODO = "auto"
CARPETA_DRIVE = "/content/drive/MyDrive/AIWEEK"
try:
    import google.colab
    detectado_colab = True
except ImportError:
    detectado_colab = False
if MODO not in {"auto", "local", "colab"}:
    raise ValueError("MODO debe ser auto, local o colab")
EN_COLAB = detectado_colab if MODO == "auto" else MODO == "colab"

def hash_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path(CARPETA_DRIVE)
    paquete_drive = DRIVE / "ai-week-colab.zip"
    if not paquete_drive.is_file():
        raise FileNotFoundError(f"Subir ai-week-colab.zip a {DRIVE}")
    paquete = Path("/content/ai-week-colab.zip")
    shutil.copy2(paquete_drive, paquete)
    huella_paquete = hash_archivo(paquete)
    raiz = Path("/content/aiweek") / huella_paquete[:12]
    if not (raiz / ".paquete_verificado").is_file():
        raiz.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paquete) as z:
            for nombre in z.namelist():
                if not (raiz / nombre).resolve().is_relative_to(raiz.resolve()):
                    raise ValueError("Ruta inválida dentro del ZIP")
            z.extractall(raiz)
        inventario = json.loads((raiz / "paquete.json").read_text(encoding="utf-8"))
        for nombre, esperado in inventario["sha256_archivos"].items():
            if hash_archivo(raiz / nombre) != esperado:
                raise ValueError(f"Archivo incompleto: {nombre}")
        (raiz / ".paquete_verificado").write_text(huella_paquete)
    dependencias = raiz / "requirements-colab.txt"
    marca = Path("/content/aiweek_dependencias_v2.sha256")
    version_deps = hash_archivo(dependencias)
    if not marca.exists() or marca.read_text() != version_deps:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(dependencias)], check=True)
        if int(importlib.metadata.version("numpy").split(".")[0]) < 2:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=2"], check=True)
        marca.write_text(version_deps)
    for nombre in ("numpy", "pandas", "transformers", "tokenizers"):
        if nombre in sys.modules and getattr(sys.modules[nombre], "__version__", None) != importlib.metadata.version(nombre):
            raise RuntimeError("Reiniciar la sesión de Colab y ejecutar desde el principio para cargar las versiones instaladas")
    RESULTADOS = DRIVE / "resultados" / huella_paquete[:12]
else:
    raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/modelos.json").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("Abrir el notebook desde la carpeta del proyecto o notebooks")
    RESULTADOS = raiz / "data/experimentos_oficiales"
RAIZ = raiz
RESULTADOS.mkdir(parents=True, exist_ok=True)
if EN_COLAB:
    corpus_drive = RESULTADOS / "corpus_procesado.zip"
    marca_corpus = raiz / ".corpus_restaurado"
    if corpus_drive.is_file():
        huella_corpus = hash_archivo(corpus_drive)
        if not marca_corpus.exists() or marca_corpus.read_text() != huella_corpus:
            carpeta_corpus = raiz / "data/processed/corpus"
            with zipfile.ZipFile(corpus_drive) as z:
                for nombre in z.namelist():
                    if not (carpeta_corpus / nombre).resolve().is_relative_to(carpeta_corpus.resolve()):
                        raise ValueError("Ruta inválida en el corpus guardado")
                z.extractall(carpeta_corpus)
            marca_corpus.write_text(huella_corpus)
os.chdir(raiz)
sys.path.insert(0, str(raiz))
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HOME"] = str(raiz / "data/cache_huggingface") if EN_COLAB else os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))
print("Entorno:", "Colab" if EN_COLAB else "local")
print("Proyecto:", raiz)
print("Resultados:", RESULTADOS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno: Colab
Proyecto: /content/aiweek/915505e96337
Resultados: /content/drive/MyDrive/AIWEEK/resultados/915505e96337


In [2]:
import gc
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scripts.evaluacion.oficial import (cargar_muestra, cargar_citaciones, metricas_recuperacion,
                                      consulta, guardar_json, huellas, abrir_experimento)
from scripts.indice.recuperacion import (Encoder, BM25, construir_indice, cargar_corpus,
                                           leer_jsonl, hash_json, sha256)
from scripts.entorno.runtime import hardware

catalogo = json.loads((raiz / "configs/modelos.json").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
precision = "float16" if device == "cuda" else "float32"
memoria_gpu = torch.cuda.get_device_properties(0).total_memory if device == "cuda" else 0
batch_size = 64 if memoria_gpu >= 20 * 2**30 else 32 if memoria_gpu >= 14 * 2**30 else 4
print(hardware() if device == "cuda" else "CPU")
print("PyTorch:", torch.__version__, "Lote:", batch_size)

NVIDIA L4, 23034 MiB, 8.9
PyTorch: 2.11.0+cu128 Lote: 64


In [3]:
encoder_elegido = "bge-m3"
tamano = 320
solapamiento = 32
hibrido = True
max_unidades = 5
reconstruir_para_verificar = False

from scripts.indice.recuperacion import cargar_indice
from scripts.evaluacion.oficial import Evidencia

ficha = next(f for f in catalogo["encoders"] if f["nombre"] == encoder_elegido)
configuracion = {"encoder": ficha, "tamano": tamano, "solapamiento": solapamiento,
                 "hibrido": hibrido, "max_unidades": max_unidades, "precision": precision,
                 "batch_size": batch_size, "huellas": huellas(raiz)}
carpeta = abrir_experimento(RESULTADOS, "recuperacion", configuracion)
indice_local = raiz / "data/index/oficial" / carpeta.name
anterior = RESULTADOS / "ultima_comparacion_encoders.json"
reutilizado = False
if anterior.exists():
    comparacion = RESULTADOS / json.loads(anterior.read_text(encoding="utf-8"))["carpeta"]
    anterior_config = json.loads((comparacion / "configuracion.json").read_text(encoding="utf-8"))
    archivo = comparacion / f"{encoder_elegido}_{tamano}.zip"
    coincide = anterior_config["huellas"] == configuracion["huellas"] and anterior_config["solapamiento"] == solapamiento
    coincide = coincide and anterior_config["precision"] == precision
    if coincide and archivo.is_file():
        with zipfile.ZipFile(archivo) as z:
            for nombre in z.namelist():
                if not (indice_local / nombre).resolve().is_relative_to(indice_local.resolve()):
                    raise ValueError("Ruta inválida en el índice")
            z.extractall(indice_local)
        recuperador = cargar_indice(indice_local, device=device)
        recuperador.bm25 = BM25([v["texto_busqueda"] for v in recuperador.ventanas]) if hibrido else None
        manifiesto = json.loads((indice_local / "manifest.json").read_text(encoding="utf-8"))
        reutilizado = True
if not reutilizado:
    config = {"encoder": ficha, "corpus": "data/processed/corpus", "tamano_tokens": tamano,
              "solapamiento_tokens": solapamiento, "device": device, "precision": precision,
              "batch_size": batch_size, "hibrido": hibrido, "salida": str(indice_local)}
    manifiesto, recuperador = construir_indice(config)
print("Índice reutilizado:", reutilizado)
if device == "cuda":
    vram = torch.cuda.memory_allocated() / 2**20
    print("VRAM del encoder:", round(vram), "MiB")
    if vram < 100:
        raise RuntimeError("El encoder no quedó en la GPU; revisar cargar_indice/Encoder en scripts/indice/recuperacion.py")
display(pd.DataFrame([{k: manifiesto[k] for k in ["unidades", "ventanas", "dimensiones", "segundos", "vram_pico_mib"]}]))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Índice reutilizado: True
VRAM del encoder: 1083 MiB


,unidades,ventanas,dimensiones,segundos,vram_pico_mib
0,43725,64073,1024,945.332355,1168.439453


## Evidencia

Cada artículo entra completo. Si el texto no identifica su norma, se añade una cabecera literal del mismo documento como pasaje separado. Se conservan los offsets de ambos y el límite de diez pasajes.

El decoder incorpora o descarta juntos el artículo y su cabecera según su ventana de contexto. Las omisiones quedan registradas.

In [4]:
muestra, entradas = cargar_muestra(raiz)
evidencia = Evidencia(raiz, raiz / "data/processed/corpus")
recuperaciones, medidas = [], []
for pregunta, entrada in zip(muestra, entradas):
    inicio = time.perf_counter()
    encontrados = recuperador.buscar(consulta(entrada), k=10)
    pasajes, avisos = evidencia.preparar(encontrados, max_unidades)
    segundos = time.perf_counter() - inicio
    evidencia.verificar(pasajes)
    recuperaciones.append({"entrada": entrada, "pasajes": pasajes, "avisos": avisos, "recuperacion_s": segundos})
    medidas.append(metricas_recuperacion(pregunta, encontrados, evidencia.documentos, evidencia.citas))
guardar_json(carpeta / "recuperaciones.json", recuperaciones)
guardar_json(carpeta / "metricas_recuperacion.json", medidas)
archivo = shutil.make_archive(str(indice_local) + "_traslado", "zip", indice_local)
shutil.copy2(archivo, carpeta / "indice.zip")
guardar_json(RESULTADOS / "recuperacion_actual.json", {
    "carpeta": str(carpeta.relative_to(RESULTADOS)),
    "sha256_recuperaciones": sha256(carpeta / "recuperaciones.json"),
    "sha256_indice": manifiesto["sha256_indice"], "huellas": configuracion["huellas"],
})
display(pd.DataFrame(medidas).groupby("formato")[["recall_normas_1", "recall_normas_5", "recall_normas_10"]].mean())
print("Preguntas:", len(recuperaciones))
print("Avisos de cabecera:", sum(len(r["avisos"]) for r in recuperaciones))
print("Guardado en:", carpeta)

,recall_normas_1,recall_normas_5,recall_normas_10
formato,,,
multiple_choice,0.269231,0.423077,0.461538
open_ended,0.250000,0.250000,0.250000
semi_open,0.166667,0.416667,0.500000


Preguntas: 50
Avisos de cabecera: 21
Guardado en: /content/drive/MyDrive/AIWEEK/resultados/915505e96337/recuperacion/20260927T205759733066Z


In [5]:
if reconstruir_para_verificar:
    config_repeticion = {"encoder": ficha, "corpus": "data/processed/corpus", "tamano_tokens": tamano,
                        "solapamiento_tokens": solapamiento, "device": device, "precision": precision,
                        "batch_size": manifiesto["configuracion"]["batch_size"], "hibrido": hibrido,
                        "salida": str(indice_local.with_name(indice_local.name + "_repeticion"))}
    repetido, _ = construir_indice(config_repeticion, recuperador.encoder)
    if repetido["sha256_indice"] != manifiesto["sha256_indice"]:
        raise RuntimeError("Las dos construcciones no producen el mismo índice")
    print("Las dos construcciones coinciden")
del recuperador
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

## Resultados y decisiones

Pendientes de ejecución. Revisar las preguntas sin respaldo y los avisos de cabecera antes de ampliar el corpus. La reconstrucción doble es opcional para esta primera medición y debe comprobarse antes de congelar la entrega.